In [17]:
import pandas as pd
import numpy as np
import yfinance as yf
from xgboost import XGBRegressor
import cvxpy as cp
import matplotlib.pyplot as plt

In [18]:
df = pd.read_csv("NIFTY50_all.csv", parse_dates=["Date"])

#Keep only relevant columns
df = df[["Date", "Symbol", "Close"]]

df.head()

,Date,Symbol,Close
0,2007-11-27,MUNDRAPORT,962.90
1,2007-11-28,MUNDRAPORT,893.90
2,2007-11-29,MUNDRAPORT,884.20
3,2007-11-30,MUNDRAPORT,921.55
4,2007-12-03,MUNDRAPORT,969.30


In [19]:
prices = df.pivot(index="Date", columns="Symbol", values="Close")
prices.head()

Symbol,ADANIPORTS,ASIANPAINT,AXISBANK,BAJAJ-AUTO,BAJAJFINSV,BAJAUTOFIN,BAJFINANCE,BHARTI,BHARTIARTL,BPCL,...,TISCO,TITAN,ULTRACEMCO,UNIPHOS,UPL,UTIBANK,VEDL,WIPRO,ZEEL,ZEETELE
Date,,,,,,,,,,,,,,,,,,,,,
2000-01-03,NaN,381.65,NaN,NaN,NaN,50.75,NaN,NaN,NaN,399.25,...,152.45,155.70,NaN,NaN,NaN,26.70,NaN,2724.20,NaN,1179.95
2000-01-04,NaN,385.55,NaN,NaN,NaN,48.10,NaN,NaN,NaN,370.50,...,150.80,147.40,NaN,NaN,NaN,26.85,NaN,2942.15,NaN,1260.65
2000-01-05,NaN,383.00,NaN,NaN,NaN,44.60,NaN,NaN,NaN,359.95,...,156.55,138.40,NaN,NaN,NaN,26.30,NaN,2990.10,NaN,1176.55
2000-01-06,NaN,377.50,NaN,NaN,NaN,45.25,NaN,NaN,NaN,380.30,...,168.25,149.50,NaN,NaN,NaN,25.95,NaN,2932.25,NaN,1115.45
2000-01-07,NaN,385.70,NaN,NaN,NaN,42.90,NaN,NaN,NaN,379.85,...,171.95,146.35,NaN,NaN,NaN,24.80,NaN,2697.70,NaN,1026.25


In [20]:
start_date = "2007-01-01"
end_date = "2019-12-31"
prices = prices.loc[start_date:end_date]
valid_stocks = prices.iloc[0].dropna().index
prices = prices[valid_stocks]

In [21]:
print(f"Dropped {len(df['Symbol'].unique()) - len(valid_stocks)} stocks because they weren't listed by {start_date}.")

Dropped 22 stocks because they weren't listed by 2007-01-01.


In [22]:
na_cols = list(prices.columns[prices.isnull().sum() > 0])
prices.drop(columns=na_cols, inplace=True)
returns = prices.pct_change().dropna()
train = returns.iloc[:int(0.8*len(returns))]
test__start_date = train.index[-1]
test = returns.iloc[int(0.8*len(returns)):]
test_end_date = test.index[-1]

In [23]:
def get_ml_forecasts(returns_df, lag):
    predicted_returns = {}
    
    for ticker in returns_df.columns:
        # Create lags of features
        lags = [returns_df[ticker].shift(i) for i in range(1, lag+1)]
        X = pd.concat(lags, axis=1).dropna()
        y = returns_df[ticker].loc[X.index]
        
        # FIX: Convert to numpy values to avoid .dtype AttributeErrors
        X_np = X.values.astype(np.float32)
        y_np = y.values.astype(np.float32)
        
        # Handle cases with no variance (flat prices)
        if np.std(y_np) == 0:
            predicted_returns[ticker] = 0.0
            continue

        model = XGBRegressor(n_estimators=50, max_depth=lag, learning_rate=0.1, verbosity=0)
        model.fit(X_np, y_np)
        
        # Predict the next expected return
        latest_features = returns_df[ticker].iloc[-lag:].values.reshape(1, -1).astype(np.float32)
        predicted_returns[ticker] = float(model.predict(latest_features)[0])
        
    return pd.Series(predicted_returns)

In [24]:
print("Predicting returns via XGBoost...")
ml_mu = get_ml_forecasts(train, 10).values  # Convert to numpy for cvxpy
n_assets = len(ml_mu)

Predicting returns via XGBoost...


In [25]:
m = np.mean(ml_mu)
lam = 20*m

In [31]:
import itertools

def brute_force(A):
    N_sub = 15
    A_sub = A[:N_sub, :N_sub]

    best_cost = float('inf')
    best_selection = None

    # Iterate through all binary combinations
    for bits in itertools.product([0, 1], repeat=N_sub):
        x = np.array(bits)
        # Calculate x^T * A * x
        cost = x.T @ A_sub @ x
        
        if cost < best_cost:
            best_cost = cost
            best_selection = x

    print(f"Brute Force Assets Selected: {np.sum(best_selection)}")

    return best_selection

In [30]:
def get_matrix(ml_mu, lam, t):
    diag = np.zeros(n_assets)
    for i in range(n_assets):
        diag[i] = -ml_mu[i] - t*lam
    A = np.zeros((n_assets, n_assets))
    for i in range(n_assets):
        for j in range(n_assets):
            if i == j:
                A[i][j] = diag[i]
            else:
                A[i][j] = 2*lam
    return A

In [32]:
A = get_matrix(ml_mu, lam = 20*m, t=10)
brute_force(A)

Brute Force Assets Selected: 3


array([1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0])

In [33]:
A = get_matrix(ml_mu, lam = 20*m, t=20)
brute_force(A)

Brute Force Assets Selected: 6


array([1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0])

In [34]:
A = get_matrix(ml_mu, lam = 20*m, t=30)
brute_force(A)

Brute Force Assets Selected: 8


array([1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0])

In [36]:
A = get_matrix(ml_mu, lam = 10*m, t=20)
brute_force(A)

Brute Force Assets Selected: 6


array([1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0])

In [28]:
def simulated_annealing(Q, num_steps=10000, T_start=10.0, T_end=0.01):
    n = Q.shape[0]
    # Start with a random selection of 10 assets
    current_state = np.zeros(n)
    current_state[np.random.choice(n, 10, replace=False)] = 1
    
    def get_cost(state):
        return state.T @ Q @ state

    current_cost = get_cost(current_state)
    
    for i in range(num_steps):
        # Linear cooling schedule
        T = T_start * (T_end / T_start) ** (i / num_steps)
        
        # Neighbor: Flip one asset (turn one off, turn another on to maintain ~10)
        new_state = current_state.copy()
        idx_to_flip = np.random.randint(0, n)
        new_state[idx_to_flip] = 1 - new_state[idx_to_flip]
        
        new_cost = get_cost(new_state)
        delta_e = new_cost - current_cost
        
        # Metropolis Criterion
        if delta_e < 0 or np.random.rand() < np.exp(-delta_e / T):
            current_state = new_state
            current_cost = new_cost
            
    return current_state

# Run the custom solver
x_sa = simulated_annealing(A)
print(f"Total Assets Selected: {np.sum(x_sa)}")

Total Assets Selected: 5.0


In [29]:
# Extract the asset names from your 'train' DataFrame columns
selected_assets = train.columns[x_sa == 1].tolist()
top_10_ranked = pd.Series(ml_mu, index=train.columns).nlargest(10).index.tolist()

print("QUBO Selection:", selected_assets)
print("Top 10 ML Ranked:", top_10_ranked)

# Check for overlap
overlap = set(selected_assets).intersection(set(top_10_ranked))
print(f"Overlap: {len(overlap)}/10 assets")

QUBO Selection: ['ASIANPAINT', 'CIPLA', 'HDFC', 'RELIANCE', 'ULTRACEMCO']
Top 10 ML Ranked: ['SHREECEM', 'HDFC', 'TECHM', 'INDUSINDBK', 'ASIANPAINT', 'EICHERMOT', 'SUNPHARMA', 'ULTRACEMCO', 'JSWSTEEL', 'TCS']
Overlap: 3/10 assets
